# 08 — Attention & Transformers

Self-attention lets each token in a sequence attend to every other token, enabling long-range dependencies without recurrence.

## Scaled Dot-Product Attention (from scratch)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

def scaled_dot_product_attention(q, k, v, mask=None):
    d_k = q.size(-1)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)   # (batch, seq_q, seq_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights @ v, weights

q = torch.randn(2, 5, 16)   # (batch, seq_len, d_k)
k = torch.randn(2, 5, 16)
v = torch.randn(2, 5, 16)
out, attn_weights = scaled_dot_product_attention(q, k, v)
out.shape, attn_weights.shape   # (2,5,16), (2,5,5)


## Multi-Head Attention (built-in)

In [ ]:
mha = nn.MultiheadAttention(embed_dim=64, num_heads=8, batch_first=True)

x = torch.randn(2, 10, 64)   # (batch, seq_len, embed_dim)
out, attn_weights = mha(query=x, key=x, value=x)   # self-attention: q=k=v
out.shape                    # (2, 10, 64)
attn_weights.shape           # (2, 10, 10) — averaged across heads


## Causal (Masked) Attention — for autoregressive models

In [ ]:
seq_len = 5
causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()  # True = masked out
# nn.MultiheadAttention expects attn_mask with True meaning "do not attend"
out, _ = mha(x[:, :seq_len], x[:, :seq_len], x[:, :seq_len], attn_mask=causal_mask)
out.shape


## Positional Encoding

Transformers have no inherent notion of order — position info must be added explicitly.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))   # not a parameter, but moves with .to(device)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

pos_enc = PositionalEncoding(d_model=64)
pos_enc(torch.randn(2, 10, 64)).shape


## TransformerEncoderLayer (built-in full block)

In [ ]:
encoder_layer = nn.TransformerEncoderLayer(
    d_model=64, nhead=8, dim_feedforward=256, dropout=0.1, batch_first=True
)
encoder = nn.TransformerEncoder(encoder_layer, num_layers=6)

x = torch.randn(2, 10, 64)
out = encoder(x)
out.shape   # (2, 10, 64)


## Full Encoder-only Transformer for Classification

In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=8, num_layers=4, num_classes=2, max_len=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x, src_key_padding_mask=None):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        x = self.encoder(x, src_key_padding_mask=src_key_padding_mask)
        pooled = x.mean(dim=1)             # mean-pool over sequence
        return self.classifier(pooled)

model = TransformerClassifier(vocab_size=10000, num_classes=2)
tokens = torch.randint(1, 10000, (4, 20))
model(tokens).shape   # (4, 2)


## Padding Mask (for variable-length sequences)

In [ ]:
tokens = torch.tensor([
    [5, 12, 8, 0, 0],     # padded with 0
    [7, 3, 9, 4, 6],
])
padding_mask = (tokens == 0)   # True where padded -> nn.Transformer* uses True = ignore
padding_mask


## Using Pretrained Transformers (HuggingFace, common in practice)

In [ ]:
# pip install transformers
from transformers import AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

inputs = tokenizer("Hello world", return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)
outputs.last_hidden_state.shape   # (1, seq_len, hidden_size)


## Common Gotchas
- `nn.MultiheadAttention` defaults to `batch_first=False` — always pass `batch_first=True` explicitly to match typical `(batch, seq, dim)` conventions.
- Attention masks in PyTorch use `True`/`-inf` to mean 'block this position' — easy to invert by mistake.
- Positional encodings must be added before the first attention layer, not after.
- Self-attention is O(n²) in sequence length — long sequences need chunking or efficient-attention variants.